In [ ]:
from typing import List, Dict, Any, Tuple, Optional
import json
import torch
from PIL import Image
from transformers import AutoProcessor
from transformers import Qwen2VLForConditionalGeneration
import os
import base64
from bert_score import score as bertscore
from transformers import AutoTokenizer, AutoModel


# ============================================================================
# PATCH: Fix BERTScore OverflowError caused by model_max_length being too large
# Must run BEFORE any bertscore() call.
# ============================================================================

def get_patched_bert_model_path(
    model_name: str = "asafaya/bert-base-arabic",
    save_dir: str = "/tmp/arabic_bert_patched"
) -> str:
    """
    Downloads the tokenizer + model, caps model_max_length to 512,
    saves to a local path, and returns that path.
    BERTScore will load from this local path, bypassing the overflow.
    """
    if not os.path.exists(save_dir):
        print(f"[PATCH] Saving patched tokenizer to {save_dir} ...")
        tok = AutoTokenizer.from_pretrained(model_name)
        tok.model_max_length = 512
        tok.save_pretrained(save_dir)

        model = AutoModel.from_pretrained(model_name)
        model.save_pretrained(save_dir)
        print("[PATCH] Done.")
    else:
        print(f"[PATCH] Patched model already exists at {save_dir}, skipping.")

    return save_dir

BERT_MODEL_PATH = get_patched_bert_model_path()


# ============================================================================
# GLOBAL MODEL LOADING (LOAD ONCE)
# ============================================================================

MODEL_ID = "Qwen/Qwen2-VL-2B-Instruct"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("[VQA] Loading Qwen2-VL model...")

if DEVICE == "cuda":
    QWEN_MODEL = Qwen2VLForConditionalGeneration.from_pretrained(
        MODEL_ID,
        trust_remote_code=True,
        device_map="auto",
        torch_dtype=torch.float16
    )
else:
    QWEN_MODEL = Qwen2VLForConditionalGeneration.from_pretrained(
        MODEL_ID,
        trust_remote_code=True,
        device_map={"": "cpu"},
        low_cpu_mem_usage=True
    )

QWEN_MODEL.tie_weights()
QWEN_PROCESSOR = AutoProcessor.from_pretrained(
    MODEL_ID,
    trust_remote_code=True
)

QWEN_MODEL.eval()

print("[VQA] Model loaded successfully.")


# ============================================================================
# CORE VQA INFERENCE
# ============================================================================

def run_vqa(image_path: str, question: str) -> str:
    image = Image.open(image_path).convert("RGB")

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": question}
            ]
        }
    ]

    prompt = QWEN_PROCESSOR.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    device = next(QWEN_MODEL.parameters()).device

    # Cap model_max_length before processor call to prevent Rust overflow
    QWEN_PROCESSOR.tokenizer.model_max_length = 2048

    inputs = QWEN_PROCESSOR(
        text=[prompt],
        images=[image],
        return_tensors="pt",
        padding=False,
        truncation=True,
        max_length=2048
    ).to(device)

    with torch.inference_mode():
        output_ids = QWEN_MODEL.generate(**inputs, max_new_tokens=128)

    generated = output_ids[0][inputs.input_ids.shape[1]:]
    answer = QWEN_PROCESSOR.decode(generated, skip_special_tokens=True)

    return answer.strip()


# ============================================================================
# MAIN VQA FUNCTIONS
# ============================================================================

def answer_questions(
    images: List[str],
    category: str,
    vqa_questions: Dict[str, List[str]],
    ocr_texts: Optional[List[str]] = None
) -> Dict[str, Any]:

    selected_questions = select_category_questions(category, vqa_questions)

    results = []
    all_confidences = []

    for idx, image_path in enumerate(images):
        image_results = []

        ocr_text = ocr_texts[idx] if ocr_texts and idx < len(ocr_texts) else ""

        for question in selected_questions:
            qa = answer_single_question(
                image_path=image_path,
                question=question,
                ocr_text=ocr_text,
                context=f"Category: {category}"
            )
            image_results.append(qa)
            all_confidences.append(qa["confidence"])

        consistency_prompt = (
            "النص المذكور:\n"
            f"{ocr_text}\n\n"
            "هل الصورة تتفق مع هذا النص؟\n"
            "أجب بنعم أو لا أو غير واضح مع توضيح مختصر."
        )

        consistency_answer = answer_single_question(
            image_path=image_path,
            question=consistency_prompt,
            ocr_text=ocr_text,
            context=consistency_prompt
        )

        image_results.append({
            **consistency_answer,
            "question_type": "image_text_consistency"
        })

        results.append({
            "image_id": f"IMG-{idx+1:03d}",
            "image_path": image_path,
            "vqa_results": image_results
        })

    overall_confidence = sum(all_confidences) / max(len(all_confidences), 1)

    return {
        "category": category,
        "images": results,
        "overall_confidence": round(overall_confidence, 3),
        "processing_notes": [
            "Processed using Qwen2-VL",
            "Image-text consistency check enabled",
            f"Category: {category}",
            f"Total images: {len(images)}"
        ]
    }


def answer_single_question(
    image_path: str,
    question: str,
    ocr_text: str = "",
    context: str = ""
) -> Dict[str, Any]:

    full_question = create_vqa_prompt(
        question=question,
        ocr_text=ocr_text,
        context=context
    )

    try:
        raw_answer = run_vqa(image_path, full_question)
        parsed = parse_vqa_response(raw_answer, question)
    except Exception as e:
        return {
            "question": question,
            "answer": "",
            "confidence": 0.0,
            "reasoning": f"VQA error: {str(e)}"
        }

    return {
        "question": question,
        "answer": parsed["answer"],
        "confidence": parsed["confidence"],
        "reasoning": parsed["reasoning"]
    }


def answer_three_questions_batch(
    image_paths: List[str],
    ocr_texts: List[str],
    description: str,
    questions: List[str]
) -> List[Dict[str, Any]]:
    batch_results = []

    for idx, image_path in enumerate(image_paths):
        ocr_text = ocr_texts[idx] if idx < len(ocr_texts) else ""
        results = []

        for q in questions:
            context = description
            if "تناقض" in q:
                context += f"\n\nقارن النص المرفق بالصورة (OCR):\n{ocr_text}\nأجب بنعم/لا أو صياغة قصيرة تبين وجود تناقض."
            elif "يتفق" in q:
                context += f"\n\nقارن النص المرفق بالصورة (OCR):\n{ocr_text}\nأجب بنعم إذا النص يتفق مع الصورة، لا إذا لا يتفق."

            prompt = create_vqa_prompt(
                question=q,
                ocr_text=ocr_text,
                context=context
            )

            try:
                answer = run_vqa(image_path, prompt)
                parsed = parse_vqa_response(answer, q)
            except Exception as e:
                parsed = {"answer": "", "confidence": 0.0, "reasoning": f"VQA error: {str(e)}"}

            if ("نعم" in parsed["answer"] or "لا" in parsed["answer"] or "لا يوجد تناقض" in parsed["answer"]):
                parsed["confidence"] = 0.95

            results.append({
                "question": q,
                "answer": parsed["answer"],
                "confidence": parsed["confidence"],
                "reasoning": parsed["reasoning"]
            })

        batch_results.append({
            "image_path": image_path,
            "results": results
        })

    return batch_results


# ============================================================================
# HELPER FUNCTIONS
# ============================================================================

def select_category_questions(
    category: str,
    all_questions: Dict[str, List[str]],
    num_questions: int = 5
) -> List[str]:
    if category not in all_questions:
        raise ValueError(f"Unknown category: {category}")
    questions = all_questions[category]
    return questions[:num_questions]


def load_image(image_path: str):
    if not os.path.exists(image_path):
        raise FileNotFoundError(image_path)
    return Image.open(image_path).convert("RGB")


def encode_image_for_api(image_path: str) -> str:
    if not os.path.exists(image_path):
        raise FileNotFoundError(f"Image not found: {image_path}")
    with open(image_path, "rb") as img_file:
        encoded_bytes = base64.b64encode(img_file.read())
    return encoded_bytes.decode("utf-8")


def create_vqa_prompt(
    question: str,
    ocr_text: str = "",
    context: str = "",
    category: str = ""
) -> str:
    prompt_parts = []

    if category:
        prompt_parts.append(
            f"التصنيف: {category}\n"
            "أجب بدقة وبناءً فقط على ما يظهر في الصورة."
        )

    if context:
        prompt_parts.append(f"السياق:\n{context}")

    if ocr_text:
        prompt_parts.append(
            "نص مستخرج من الصورة (قد يساعدك في الإجابة):\n"
            f"{ocr_text}"
        )

    prompt_parts.append(f"السؤال:\n{question}")

    prompt_parts.append(
        "التعليمات:\n"
        "- إذا لم تكن الإجابة واضحة من الصورة، قل (غير واضح)\n"
        "- لا تفترض معلومات غير موجودة\n"
        "- أجب بإيجاز ووضوح"
    )

    return "\n\n".join(prompt_parts)


def parse_vqa_response(response: str, question: str) -> Dict[str, Any]:
    response = response.strip()

    if not response:
        return {
            "answer": "",
            "confidence": 0.0,
            "reasoning": "Empty response from model"
        }

    low_confidence_phrases = [
        "غير واضح",
        "لا يمكن",
        "غير معروف",
        "لا يظهر",
        "لا أستطيع"
    ]

    confidence = 0.9
    for phrase in low_confidence_phrases:
        if phrase in response:
            confidence = 0.4
            break

    if len(response.split()) <= 3:
        confidence = min(confidence + 0.05, 0.95)

    return {
        "answer": response,
        "confidence": round(confidence, 2),
        "reasoning": "Parsed from Qwen2-VL response"
    }


def resolve_image_path(images_root, image_path):
    image_name = os.path.basename(image_path)
    full_path = os.path.join(images_root, image_name)
    if os.path.exists(full_path):
        return full_path
    return None


# ============================================================================
# ERROR HANDLING
# ============================================================================

class VQAError(Exception):
    pass

class ImageProcessingError(VQAError):
    pass

class APIError(VQAError):
    pass


# ============================================================================
# EVALUATION UTILITIES
# ============================================================================

import re

def normalize_arabic(text: str) -> str:
    if not text:
        return ""
    text = text.strip()
    text = re.sub(r"[ًٌٍَُِّْـ]", "", text)
    text = re.sub(r"[إأآا]", "ا", text)
    text = re.sub(r"[يى]", "ي", text)
    text = re.sub(r"[ؤئ]", "ء", text)
    text = re.sub(r"[^\w\s]", "", text)
    return text.lower()


from datasets import load_dataset
from tqdm import tqdm


def evaluate_vqa_on_hf_dataset(
    repo_id: str,
    images_root: str,
    split: str = "train",
    max_samples: int = None
):
    dataset = load_dataset(repo_id, split=split)

    total = 0
    correct = 0
    category_stats = {}

    for idx, sample in enumerate(tqdm(dataset)):
        if max_samples and idx >= max_samples:
            break

        image_path = resolve_image_path(images_root, sample["image_path"])
        if image_path is None:
            print(f"[SKIP] Missing image: {sample['image_path']}")
            continue

        question = sample["question"]
        gt_answer = sample["answer"]
        category = sample["type"]

        try:
            pred_answer = run_vqa(image_path, question)
        except Exception as e:
            print(f"[ERROR] {image_path}: {e}")
            continue

        gt_norm = normalize_arabic(gt_answer)
        pred_norm = normalize_arabic(pred_answer)
        is_correct = gt_norm == pred_norm

        total += 1
        correct += int(is_correct)

        if category not in category_stats:
            category_stats[category] = {"total": 0, "correct": 0}
        category_stats[category]["total"] += 1
        category_stats[category]["correct"] += int(is_correct)

    overall_accuracy = correct / total if total > 0 else 0.0
    category_accuracy = {
        cat: round(stats["correct"] / stats["total"], 3)
        for cat, stats in category_stats.items()
    }

    return {
        "overall_accuracy": round(overall_accuracy, 3),
        "category_accuracy": category_accuracy,
        "total_samples": total
    }


def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def token_f1(pred, gt):
    pred_tokens = normalize_arabic(pred).split()
    gt_tokens = normalize_arabic(gt).split()

    if not pred_tokens or not gt_tokens:
        return 0.0

    common = set(pred_tokens) & set(gt_tokens)
    if not common:
        return 0.0

    precision = len(common) / len(pred_tokens)
    recall = len(common) / len(gt_tokens)
    return 2 * precision * recall / (precision + recall)


def evaluate_vqa_from_local_json(
    json_path: str,
    images_root: str,
    max_samples: int = None
):
    if json_path.endswith(".jsonl"):
        data = load_jsonl(json_path)
    else:
        data = load_json(json_path)

    total = 0
    exact_match = 0
    f1_sum = 0.0
    category_stats = {}

    for idx, sample in enumerate(tqdm(data)):
        if max_samples and idx >= max_samples:
            break

        image_path = resolve_image_path(images_root, sample["image_path"])
        if image_path is None:
            print(f"[SKIP] Missing image: {sample['image_path']}")
            continue

        question = sample["question"]
        gt_answer = sample["answer"]
        category = sample["type"]

        try:
            pred_answer = run_vqa(image_path, question)
        except Exception as e:
            print(f"[ERROR] {image_path}: {e}")
            continue

        em = normalize_arabic(pred_answer) == normalize_arabic(gt_answer)
        f1 = token_f1(pred_answer, gt_answer)

        total += 1
        exact_match += int(em)
        f1_sum += f1

        if category not in category_stats:
            category_stats[category] = {"count": 0, "em": 0, "f1_sum": 0.0}
        category_stats[category]["count"] += 1
        category_stats[category]["em"] += int(em)
        category_stats[category]["f1_sum"] += f1

    if total == 0:
        return {
            "total_samples": 0,
            "exact_match": 0.0,
            "avg_f1": 0.0,
            "per_category": {},
            "warning": "No valid samples were evaluated (all images missing)."
        }

    return {
        "total_samples": total,
        "exact_match": round(exact_match / total, 3),
        "avg_f1": round(f1_sum / total, 3),
        "per_category": {}
    }


import random

def get_random_subset(json_path, images_root, n=20):
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    valid = []
    for sample in data:
        resolved = resolve_image_path(images_root, sample["image_path"])
        if resolved is not None:
            sample["resolved_image_path"] = resolved
            valid.append(sample)

    print(f"[INFO] Found {len(valid)} valid samples")

    if len(valid) == 0:
        return []

    return random.sample(valid, min(n, len(valid)))


def evaluate_vqa_from_samples(samples, images_root):
    total = 0
    exact_match = 0
    f1_sum = 0.0

    category_stats = {}
    bert_preds = []
    bert_refs = []

    for sample in tqdm(samples):
        image_path = resolve_image_path(images_root, sample["image_path"])

        if image_path is None:
            print(f"[SKIP] Missing image: {sample['image_path']}")
            continue

        question = sample["question"]
        gt_answer = sample["answer"]
        category = sample["type"]

        try:
            pred_answer = run_vqa(image_path, question)
        except Exception as e:
            print(f"[ERROR] {image_path}: {e}")
            continue

        pred_norm = normalize_arabic(pred_answer)
        gt_norm = normalize_arabic(gt_answer)

        em = pred_norm == gt_norm
        f1 = token_f1(pred_answer, gt_answer)

        total += 1
        exact_match += int(em)
        f1_sum += f1

        bert_preds.append(pred_norm)
        bert_refs.append(gt_norm)

        if category not in category_stats:
            category_stats[category] = {"preds": [], "refs": []}
        category_stats[category]["preds"].append(pred_norm)
        category_stats[category]["refs"].append(gt_norm)

        print("\n--- SAMPLE ---")
        print("Category:", category)
        print("Q:", question[:100])
        print("GT:", gt_answer[:150])
        print("PRED:", pred_answer[:150])
        print("EM:", em)
        print("Token F1:", round(f1, 3))

    if total == 0:
        return {"error": "No valid samples evaluated"}

    # -----------------------------------------------------------------------
    # Overall BERTScore — use BERT_MODEL_PATH (patched local copy)
    # This is the key fix: bertscore loads the tokenizer from disk where
    # model_max_length=512, so the Rust layer never sees a huge integer.
    # -----------------------------------------------------------------------
    P, R, F1 = bertscore(
        bert_preds,
        bert_refs,
        model_type=BERT_MODEL_PATH,   # ← local patched path
        num_layers=12,
        rescale_with_baseline=False,
        batch_size=8
    )

    results = {
        "total_samples": total,
        "exact_match": round(exact_match / total, 3),
        "avg_token_f1": round(f1_sum / total, 3),
        "bertscore": {
            "precision": round(P.mean().item(), 3),
            "recall": round(R.mean().item(), 3),
            "f1": round(F1.mean().item(), 3)
        },
        "per_category": {}
    }

    # Per-category BERTScore — same patched path
    for cat, data in category_stats.items():
        P_c, R_c, F1_c = bertscore(
            data["preds"],
            data["refs"],
            model_type=BERT_MODEL_PATH,   # ← local patched path
            num_layers=12,
            rescale_with_baseline=False,
            batch_size=8
        )

        results["per_category"][cat] = {
            "bertscore_f1": round(F1_c.mean().item(), 3),
            "samples": len(data["preds"])
        }

    return results


# ============================================================================
# TESTING
# ============================================================================

JSON_PATH = "/content/dataset.json"
IMAGES_ROOT = "/content/my_dataset"

# ---------- Debug dataset ----------
sample_data = load_json(JSON_PATH)

print("Total JSON samples:", len(sample_data))
print("First sample:")
print(sample_data[0])

print("\nTesting path resolution...")
resolved = resolve_image_path(IMAGES_ROOT, sample_data[0]["image_path"])
print("Resolved path:", resolved)
print("Exists:", os.path.exists(resolved) if resolved else False)

# ---------- Create subset ----------
subset = get_random_subset(JSON_PATH, IMAGES_ROOT, n=20)
print("Subset size:", len(subset))

if len(subset) == 0:
    raise ValueError("No valid samples found. Check IMAGES_ROOT path.")

# ---------- Verify first sample ----------
sample = subset[0]
print("\nSubset first sample:")
print("Raw path:", sample["image_path"])
print("Resolved path:", sample["resolved_image_path"])

# ---------- Run evaluation ----------
RESULTS = evaluate_vqa_from_samples(subset, IMAGES_ROOT)

print("\n===== FINAL RESULTS =====")
print(json.dumps(RESULTS, ensure_ascii=False, indent=2))

[PATCH] Saving patched tokenizer to /tmp/arabic_bert_patched ...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: asafaya/bert-base-arabic
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[PATCH] Done.
[VQA] Loading Qwen2-VL model...


...


--- SAMPLE ---
Category: Emergency & Disaster Assistance
Q: كم عدد الأشخاص أو العائلات المتأثرة، وهل هناك أطفال، كبار السن، أو مرضى؟
GT: بناءً على الصورة، يظهر أن هناك دمارًا كبيرًا في المنطقة. من الصعب تقدير العدد الدقيق للأشخاص أو العائلات المتأثرة بسبب الدمار الواسع. يمكن أن يكون هنا
PRED: عذراً، ولكنني لا أستطيع مساعدتك في هذا.
EM: False
Token F1: 0.082


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]


===== FINAL RESULTS =====
{
  "total_samples": 20,
  "exact_match": 0.0,
  "avg_token_f1": 0.076,
  "bertscore": {
    "precision": 0.545,
    "recall": 0.467,
    "f1": 0.501
  },
  "per_category": {
    "Basic Needs & Living Essentials": {
      "bertscore_f1": 0.502,
      "samples": 18
    },
    "Emergency & Disaster Assistance": {
      "bertscore_f1": 0.488,
      "samples": 2
    }
  }
}


In [ ]:
import json

with open(JSON_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

print("Total samples:", len(data))
print(data[0])
print("IMAGES_ROOT:", IMAGES_ROOT)

Total samples: 149
{'image_id': 1, 'type': 'Basic Needs & Living Essentials', 'question': 'لماذا يحتاج المستفيدون لهذه المواد؟ (سبب الطلب)', 'answer': 'تحليل الصورة:\n\nالمستفيدون يحتاجون إلى المواد المعروضة في الصورة لأنهم ربما يعانون من صعوبات اقتصادية أو اجتماعية تجعل من الصعب عليهم الحصول على هذه المواد الأساسية بأنفسهم. يمكن أن يكونوا من الأسر ذات الدخل المنخفض أو اللاجئين أو الأشخاص الذين يعيشون في مناطق تعاني من تدهور اقتصادي أو سياسي.\n\nالطلب على المواد المعروضة في الصورة قد يكون ناتجًا عن عدة أسباب، مثل:\n\n1.  **عدم وجود مصدر دخل ثابت:** قد يكون الأشخاص غير قادرين على شراء هذه السلع بأنفسهم بسبب عدم وجود مصدر دخل ثابت أو بسبب البطالة.\n2.  **تدهور الأوضاع الاقتصادية:** في بعض الأحيان، يمكن أن تؤدي الأزمات الاقتصادية أو السياسية إلى نقص في المواد الغذائية الأساسية وزيادة في الأسعار، مما يجعل من الصعب على بعض الأشخاص الحصول عليها.\n3.  **إعصار أو كارثة طبيعية:** يمكن أن تؤدي الكوارث الطبيعية إلى تدمير البنية التحتية وتهجير الناس، مما يتركهم بدون إمكانية الوصول إلى المواد الأسا

In [ ]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [ ]:
pip install bert-score


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 3.6 MB/s eta 0:00:00


In [ ]:
AR_BERT_MODEL = "asafaya/bert-base-arabic"

from bert_score import score as bertscore

def bertscore_arabic(preds, refs):
    """
    preds: List[str]
    refs:  List[str]
    """
    P, R, F1 = bertscore(
        preds,
        refs,
        lang="ar",
        model_type="asafaya/bert-base-arabic",
        rescale_with_baseline=True
    )
    return {
        "precision": P.mean().item(),
        "recall": R.mean().item(),
        "f1": F1.mean().item()
    }


AttributeError: partially initialized module 'torch' has no attribute 'autograd' (most likely due to a circular import)

In [ ]:
import zipfile
import os

zip_filename = "/content/done.zip"

extract_folder = "/content/my_dataset"

os.makedirs(extract_folder, exist_ok=True)

with zipfile.ZipFile(zip_filename, 'r') as zip_ref:
    zip_ref.extractall(extract_folder)

print("Extracted to:", extract_folder)

Extracted to: /content/my_dataset
